# Football Analysis — Tracking trên Colab GPU

Chỉ chạy **YOLO + BoT-SORT + ReID** trên GPU. Phần còn lại (camera, team, tactical, render) chạy **local**.

## Chuẩn bị trên Google Drive
Tạo thư mục `MyDrive/football_analysis/` và đặt sẵn:
```
football_analysis/
├── best.pt
└── input_video.mp4    ← video cần tracking
```

## Quy trình
1. Chạy các cell bên dưới trên Colab (GPU)
2. Stub tự lưu lên Drive + tải về máy
3. Copy `track_stubs.pkl` → `stubs/track_stubs.pkl` trên local
4. Copy **cùng video** → `input_videos/input_video.mp4` trên local
5. Chạy backend local + frontend (xóa `.env.local` nếu không dùng ngrok)

API/CLI tự nhận stub và **bỏ qua bước tracking**.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/PhuongAnh777/football_analysis.git"
BRANCH = "main"

if os.path.isdir("football_analysis"):
    %cd football_analysis
    !git pull
else:
    !git clone -b {BRANCH} {REPO_URL}
    %cd football_analysis

In [ ]:
!pip install -q ultralytics opencv-python numpy pandas scikit-learn scipy matplotlib

In [ ]:
import os
from google.colab import drive

# ── Sửa path nếu thư mục Drive khác ──────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/football_analysis"
DRIVE_MODEL  = f"{DRIVE_ROOT}/best.pt"
DRIVE_VIDEO  = f"{DRIVE_ROOT}/input_video.mp4"

MODEL_PATH = "models/best.pt"
VIDEO_PATH = "input_videos/input_video.mp4"

os.makedirs("models", exist_ok=True)
os.makedirs("input_videos", exist_ok=True)
os.makedirs("stubs", exist_ok=True)

drive.mount("/content/drive")

for src, dst, label in [
    (DRIVE_MODEL, MODEL_PATH, "Model"),
    (DRIVE_VIDEO, VIDEO_PATH, "Video"),
]:
    if not os.path.exists(src):
        raise FileNotFoundError(f"{label} không thấy trên Drive: {src}")
    !cp "{src}" "{dst}"
    size_mb = os.path.getsize(dst) / (1024 * 1024)
    print(f"{label} OK: {dst} ({size_mb:.1f} MB)")

In [ ]:
# Fallback: upload thủ công nếu chưa có trên Drive (bỏ qua cell này nếu Drive OK)
import os
from google.colab import files

if os.path.exists("input_videos/input_video.mp4") and os.path.exists("models/best.pt"):
    print("Model + video đã sẵn sàng từ Drive.")
else:
    if not os.path.exists("models/best.pt"):
        print("Upload best.pt:")
        files.upload()
        !mv best.pt models/best.pt
    if not os.path.exists("input_videos/input_video.mp4"):
        print("Upload video MP4/AVI:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        !mv "{name}" input_videos/input_video.mp4

In [ ]:
!python scripts/run_tracking.py input_videos/input_video.mp4 --stub stubs/track_stubs.pkl

In [ ]:
import os
from google.colab import files

STUB_LOCAL = "stubs/track_stubs.pkl"
STUB_DRIVE = "/content/drive/MyDrive/football_analysis/track_stubs.pkl"

size_mb = os.path.getsize(STUB_LOCAL) / (1024 * 1024)
print(f"Stub: {STUB_LOCAL} ({size_mb:.1f} MB)")

!cp "{STUB_LOCAL}" "{STUB_DRIVE}"
print(f"Đã lưu lên Drive: {STUB_DRIVE}")

files.download(STUB_LOCAL)
print("Đã tải stub — copy vào stubs/track_stubs.pkl trên máy local.")

## Local — copy từ Drive hoặc file vừa tải

Trên máy local, lấy từ Drive (hoặc file download):
- `track_stubs.pkl` → `stubs/track_stubs.pkl`
- `input_video.mp4` → `input_videos/input_video.mp4`

Rồi chạy backend + frontend như bình thường.